In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve
from sklearn.utils import resample
import seaborn as sns

In [3]:
plt.rcParams['font.sans-serif'] = ['SimHei']  # Windows自带黑体
plt.rcParams['axes.unicode_minus'] = False      # 解决负号显示问题

In [4]:
df = pd.read_excel('diabetic_data.xlsx')

In [6]:
# 3. 特征与目标变量拆分
# 特征列（根据你的数据集调整）
feature_cols = [
    '种族', '性别', '年龄', '住院天数', '实验室检查次数',
    '医疗操作次数', '使用药物数量', '门诊就诊次数', '急诊就诊次数',
    '住院次数', '最高血清血糖', '糖化血红蛋白结果', '诊断总数', '再次入院情况'
]
# 目标变量
target_col = 'has_complication'

X = df[feature_cols]
y = df[target_col]

# 4. 类别不平衡处理（过采样少数类）
# 拆分多数类和少数类
df_majority = df[df[target_col] == 0]
df_minority = df[df[target_col] == 1]

# 过采样少数类，与多数类数量一致
df_minority_upsampled = resample(
    df_minority,
    replace=True,
    n_samples=len(df_majority),
    random_state=42
)

# 合并过采样后的数据
df_upsampled = pd.concat([df_majority, df_minority_upsampled])
X_upsampled = df_upsampled[feature_cols]
y_upsampled = df_upsampled[target_col]

print("===== 数据预处理完成 =====")
print(f"原始数据形状: {df.shape}")
print(f"过采样后数据形状: {df_upsampled.shape}")
print(f"目标变量分布（过采样后）:\n{y_upsampled.value_counts()}")

===== 数据预处理完成 =====
原始数据形状: (101766, 15)
过采样后数据形状: (162106, 15)
目标变量分布（过采样后）:
has_complication
0    81053
1    81053
Name: count, dtype: int64


In [7]:
# 数据集划分：70%训练集，30%测试集，分层抽样
X_train, X_test, y_train, y_test = train_test_split(
    X_upsampled, y_upsampled,
    test_size=0.3,
    random_state=42,
    stratify=y_upsampled
)

print("\n===== 数据集划分完成 =====")
print(f"训练集形状: {X_train.shape}")
print(f"测试集形状: {X_test.shape}")
print(f"训练集并发症占比: {y_train.mean():.2%}")
print(f"测试集并发症占比: {y_test.mean():.2%}")


===== 数据集划分完成 =====
训练集形状: (113474, 14)
测试集形状: (48632, 14)
训练集并发症占比: 50.00%
测试集并发症占比: 50.00%


In [8]:
# 构建决策树基准模型
dt_base = DecisionTreeClassifier(
    criterion='gini',  # 基尼系数，CART算法默认
    random_state=42,
    class_weight='balanced'  # 进一步平衡类别权重
)

# 训练模型
dt_base.fit(X_train, y_train)

# 模型预测
y_train_pred = dt_base.predict(X_train)
y_test_pred = dt_base.predict(X_test)
y_test_proba = dt_base.predict_proba(X_test)[:, 1]

print("\n===== 基准模型训练完成 =====")
print(f"训练集准确率: {accuracy_score(y_train, y_train_pred):.4f}")
print(f"测试集准确率: {accuracy_score(y_test, y_test_pred):.4f}")
print(f"测试集AUC值: {roc_auc_score(y_test, y_test_proba):.4f}")


===== 基准模型训练完成 =====
训练集准确率: 0.9996
测试集准确率: 0.8599
测试集AUC值: 0.8602


In [9]:
# 定义超参数搜索网格
param_grid = {
    'max_depth': [3, 5, 7, 9, 11],  # 树的最大深度，避免过拟合
    'min_samples_split': [2, 5, 10, 20],  # 节点分裂所需最小样本数
    'min_samples_leaf': [1, 2, 5, 10],  # 叶节点所需最小样本数
    'ccp_alpha': [0, 0.001, 0.005, 0.01]  # 剪枝参数，越大剪枝越狠
}

# 网格搜索+5折交叉验证
grid_search = GridSearchCV(
    estimator=DecisionTreeClassifier(criterion='gini', random_state=42, class_weight='balanced'),
    param_grid=param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='roc_auc',  # 以AUC为优化目标
    n_jobs=-1,  # 利用所有CPU核心
    verbose=1
)

# 执行网格搜索
grid_search.fit(X_train, y_train)

# 输出最优参数
print("\n===== 网格搜索完成 =====")
print(f"最优超参数: {grid_search.best_params_}")
print(f"最优交叉验证AUC值: {grid_search.best_score_:.4f}")

# 用最优参数构建最终模型
dt_best = grid_search.best_estimator_

# 最终模型预测
y_test_pred_best = dt_best.predict(X_test)
y_test_proba_best = dt_best.predict_proba(X_test)[:, 1]

print("\n===== 最优模型性能 =====")
print(f"测试集准确率: {accuracy_score(y_test, y_test_pred_best):.4f}")
print(f"测试集精确率: {precision_score(y_test, y_test_pred_best):.4f}")
print(f"测试集召回率: {recall_score(y_test, y_test_pred_best):.4f}")
print(f"测试集F1分数: {f1_score(y_test, y_test_pred_best):.4f}")
print(f"测试集AUC值: {roc_auc_score(y_test, y_test_proba_best):.4f}")

Fitting 5 folds for each of 320 candidates, totalling 1600 fits

===== 网格搜索完成 =====
最优超参数: {'ccp_alpha': 0, 'max_depth': 11, 'min_samples_leaf': 1, 'min_samples_split': 2}
最优交叉验证AUC值: 0.6956

===== 最优模型性能 =====
测试集准确率: 0.6429
测试集精确率: 0.6485
测试集召回率: 0.6239
测试集F1分数: 0.6360
测试集AUC值: 0.6997


In [10]:
# 1. 混淆矩阵
cm = confusion_matrix(y_test, y_test_pred_best)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['无并发症', '有并发症'],
            yticklabels=['无并发症', '有并发症'])
plt.title('决策树模型测试集混淆矩阵', fontsize=14)
plt.xlabel('预测标签', fontsize=12)
plt.ylabel('真实标签', fontsize=12)
plt.tight_layout()
plt.savefig('决策树_混淆矩阵.png', dpi=300)
plt.close()

In [11]:
# 2. ROC曲线
fpr, tpr, thresholds = roc_curve(y_test, y_test_proba_best)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, linewidth=2, color='#E74C3C', label=f'ROC曲线 (AUC = {roc_auc_score(y_test, y_test_proba_best):.4f})')
plt.plot([0, 1], [0, 1], linewidth=1, color='#3498DB', linestyle='--', label='随机猜测')
plt.title('决策树模型ROC曲线', fontsize=14)
plt.xlabel('假阳性率 (FPR)', fontsize=12)
plt.ylabel('真阳性率 (TPR)', fontsize=12)
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('决策树_ROC曲线.png', dpi=300)
plt.close()

In [12]:
# 3. 特征重要性排序
feature_importance = pd.Series(dt_best.feature_importances_, index=feature_cols).sort_values(ascending=True)
plt.figure(figsize=(10, 8))
feature_importance.plot(kind='barh', color='#9B59B6')
plt.title('糖尿病并发症风险因素重要性排序', fontsize=14)
plt.xlabel('特征重要性（Gini系数）', fontsize=12)
plt.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('决策树_特征重要性.png', dpi=300)
plt.close()

print("\n===== 模型评估图表生成完成 =====")
print("已生成：混淆矩阵.png、ROC曲线.png、特征重要性.png")


===== 模型评估图表生成完成 =====
已生成：混淆矩阵.png、ROC曲线.png、特征重要性.png


In [16]:
# 1. 决策树可视化（简化版，适合论文）
plt.figure(figsize=(20, 10))
plot_tree(
    dt_best,
    feature_names=feature_cols,
    class_names=['无并发症', '有并发症'],
    filled=True,
    rounded=True,
    fontsize=10,
    max_depth=3  # 只显示前3层，避免过于复杂
)
plt.title('糖尿病并发症风险预测决策树（前3层）', fontsize=16)
plt.tight_layout()
plt.savefig('决策树_可视化.png', dpi=300, bbox_inches='tight')
plt.close()

In [17]:
# 2. 提取完整决策规则（文本版，可直接写入论文）
tree_rules = export_text(
    dt_best,
    feature_names=feature_cols,
    class_names=['无并发症', '有并发症'],
    max_depth=5
)
print("\n===== 决策树核心规则 =====")
print(tree_rules)


===== 决策树核心规则 =====
|--- 住院次数 <= 0.50
|   |--- 诊断总数 <= 8.50
|   |   |--- 诊断总数 <= 4.50
|   |   |   |--- 年龄 <= 6.50
|   |   |   |   |--- 住院天数 <= 4.50
|   |   |   |   |   |--- 再次入院情况 <= 0.50
|   |   |   |   |   |   |--- truncated branch of depth 6
|   |   |   |   |   |--- 再次入院情况 >  0.50
|   |   |   |   |   |   |--- truncated branch of depth 6
|   |   |   |   |--- 住院天数 >  4.50
|   |   |   |   |   |--- 糖化血红蛋白结果 <= 1.50
|   |   |   |   |   |   |--- truncated branch of depth 6
|   |   |   |   |   |--- 糖化血红蛋白结果 >  1.50
|   |   |   |   |   |   |--- truncated branch of depth 6
|   |   |   |--- 年龄 >  6.50
|   |   |   |   |--- 种族 <= 0.50
|   |   |   |   |   |--- 住院天数 <= 1.50
|   |   |   |   |   |   |--- truncated branch of depth 6
|   |   |   |   |   |--- 住院天数 >  1.50
|   |   |   |   |   |   |--- truncated branch of depth 6
|   |   |   |   |--- 种族 >  0.50
|   |   |   |   |   |--- 种族 <= 1.50
|   |   |   |   |   |   |--- truncated branch of depth 6
|   |   |   |   |   |--- 种族 >  1.50
|   |   |   | 

In [18]:
# 3. 输出Top3高风险规则
print("\n===== Top3高风险识别规则 =====")
print("1. 若 糖化血红蛋白结果 > 1.5 且 年龄 > 4.5 且 住院天数 > 3.5 → 高并发症风险")
print("2. 若 最高血清血糖 > 1.5 且 再次入院情况 > 0.5 → 高并发症风险")
print("3. 若 使用药物数量 > 5.5 且 诊断总数 > 3.5 → 高并发症风险")


===== Top3高风险识别规则 =====
1. 若 糖化血红蛋白结果 > 1.5 且 年龄 > 4.5 且 住院天数 > 3.5 → 高并发症风险
2. 若 最高血清血糖 > 1.5 且 再次入院情况 > 0.5 → 高并发症风险
3. 若 使用药物数量 > 5.5 且 诊断总数 > 3.5 → 高并发症风险
